# Notebook 2 — Rule-Based Pre-Parser

Pure Python. No ML. Goal: classify 80-90% of real notes without touching Qwen.

**Exit criteria:** 85%+ of clear cases classified correctly. Ambiguous cases correctly flagged.

## 1. Amount Parser

Handles: `5k`, `5K`, `1.5L`, `500`, `5,000`

In [1]:

import re

def parse_amount(text):
    """Extract amount from text. Returns float or None."""
    text = text.strip().replace(',', '')
    
    # Match patterns like 5k, 5K, 5.5k
    m = re.match(r'^(\d+\.?\d*)\s*[kK]$', text)
    if m:
        return float(m.group(1)) * 1000
    
    # Match patterns like 1.5L, 2L
    m = re.match(r'^(\d+\.?\d*)\s*[lL]$', text)
    if m:
        return float(m.group(1)) * 100000
    
    # Plain number
    m = re.match(r'^(\d+\.?\d*)$', text)
    if m:
        return float(m.group(1))
    
    return None

# Test
tests = [('10k', 10000), ('2.5K', 2500), ('1.55L', 155000), ('50', 50), ('5,000', 5000), ('65.1', 65.1), ('11', 11.0)]
for t, expected in tests:
    result = parse_amount(t)
    status = '✓' if result == expected else f'✗ got {result}'
    print(f"{t:10} → {result:10} {status}")


10k        →    10000.0 ✓
2.5K       →     2500.0 ✓
1.55L      →   155000.0 ✓
50         →       50.0 ✓
5,000      →     5000.0 ✓
65.1       →       65.1 ✓
11         →       11.0 ✓


## 2. (removed) Category Tagger

Categories were dropped — they added a keyword maintenance burden without paying for themselves for monthly tracking. Description is stored verbatim and ad-hoc filters use SQL `LIKE` (e.g. `WHERE description LIKE '%petrol%'`).

## 3. Core Parser Logic

In [ ]:
from datetime import datetime

# ── Person whitelist ────────────────────────────────────────
# Single source of truth. Managed via chat commands ADD_PERSON / REMOVE_PERSON / MODIFY_PERSON.
# In Notebook 5 this is loaded from the `persons` table; here we hardcode for parser tests.
KNOWN_PERSONS = {'jeevi', 'prani', 'murugan', 'maddy', 'thenna'}

# Ledger keywords
GAVE_KEYWORDS    = r'\b(gave|give|given|lent|sent|advanced)\b'
RECEIVED_KEYWORDS = r'\b(got|received|returned|paid back|paid me back|gave me)\b'

# Person command prefixes (case-insensitive). Match the *whole* line — never split on commas.
PERSON_CMD_RE = re.compile(
    r'^(ADD_PERSON|REMOVE_PERSON|MODIFY_PERSON)\s*:\s*(.+)$',
    re.IGNORECASE,
)

def get_month(date_str):
    """Extract YYYY-MM from ISO date string."""
    return date_str[:7]

def find_amount_in_tokens(tokens):
    for i, token in enumerate(tokens):
        amt = parse_amount(token)
        if amt is not None:
            return amt, i
    return None, -1

def parse_person_command(text):
    """If text is ADD_PERSON / REMOVE_PERSON / MODIFY_PERSON, return a structured dict.
    Otherwise None."""
    m = PERSON_CMD_RE.match(text.strip())
    if not m:
        return None
    op = m.group(1).upper()
    payload = m.group(2).strip().lower()
    if op == 'MODIFY_PERSON':
        parts = payload.split()
        if len(parts) != 2:
            return {'type': 'person_command', 'op': op, 'error': 'Expected: MODIFY_PERSON: oldname newname',
                    'raw': text}
        return {'type': 'person_command', 'op': op, 'old_name': parts[0], 'new_name': parts[1], 'raw': text}
    return {'type': 'person_command', 'op': op, 'name': payload, 'raw': text}

def parse_single(text, today):
    """Parse a single note entry. Returns a dict."""
    text = text.strip()
    if not text:
        return None
    
    text_lower = text.lower()

    # ── PERSON COMMAND (highest priority) ────────────────────────
    cmd = parse_person_command(text)
    if cmd:
        return cmd

    # ── LEDGER: gift keyword → expense override ──────────────────
    if re.search(r'\bgift\b', text_lower):
        tokens = text.split()
        amt, _ = find_amount_in_tokens(tokens)
        if amt:
            desc = re.sub(r'\b\d+[kKlL]?\b', '', text).replace('gift', '').strip()
            return {'type': 'expense', 'amount': amt, 'description': desc or 'gift',
                    'date': today, 'month': get_month(today), 'raw': text}

    # ── LEDGER: received direction ────────────────────────────────
    if re.search(RECEIVED_KEYWORDS, text_lower):
        person_match = re.search(r'from\s+([a-zA-Z]+)', text_lower)
        if not person_match:
            person_match = re.search(r'^([a-zA-Z]+)\s+(?:returned|paid)', text_lower)
        
        tokens = text.split()
        amt, _ = find_amount_in_tokens(tokens)
        
        if amt and person_match:
            person = person_match.group(1).lower().strip()
            return {'type': 'ledger', 'person': person, 'amount': amt,
                    'direction': 'received', 'note': None, 'date': today, 'raw': text,
                    'unknown_person': person not in KNOWN_PERSONS}

    # ── LEDGER: gave direction ────────────────────────────────────
    if re.search(GAVE_KEYWORDS, text_lower):
        gave_match = re.search(r'(?:gave|give|given|lent|sent|advanced)\s+([a-zA-Z]+)', text_lower)
        person_gave_match = re.search(r'^([a-zA-Z]+)\s+(?:gave|give)', text_lower)
        
        tokens = text.split()
        amt, _ = find_amount_in_tokens(tokens)
        
        if amt:
            if person_gave_match:
                person = person_gave_match.group(1).lower().strip()
                return {'type': 'ledger', 'person': person, 'amount': amt,
                        'direction': 'received', 'note': None, 'date': today, 'raw': text,
                        'unknown_person': person not in KNOWN_PERSONS}
            elif gave_match:
                person = gave_match.group(1).lower().strip()
                return {'type': 'ledger', 'person': person, 'amount': amt,
                        'direction': 'gave', 'note': None, 'date': today, 'raw': text,
                        'unknown_person': person not in KNOWN_PERSONS}

    # ── WEIGHT: STRICT — only KNOWN_PERSONS + number < 150 ───────
    # No more "weight_confirm" fallback — unknown name + number falls through to expense.
    for name in KNOWN_PERSONS:
        pattern = rf'\b{name}\b\s+(\d+\.?\d*)|\b(\d+\.?\d*)\s+\b{name}\b'
        m = re.search(pattern, text_lower)
        if m:
            raw_num = m.group(1) or m.group(2)
            weight = float(raw_num)
            if weight < 150:
                note_match = re.search(rf'\b{name}\b\s+\d+\.?\d*\s*(.*)', text_lower)
                note = note_match.group(1).strip() if note_match and note_match.group(1).strip() else None
                return {'type': 'weight', 'person': name, 'weight': weight,
                        'note': note, 'date': today, 'raw': text}

    # ── EXPENSE: amount + description (no category) ──────────────
    tokens = re.split(r'\s+', text.strip())
    amt, amt_idx = find_amount_in_tokens(tokens)
    
    if amt is not None:
        desc_tokens = [t for i, t in enumerate(tokens) if i != amt_idx]
        desc = ' '.join(desc_tokens).strip(' -+')
        if not desc:
            desc = 'misc'
        
        # Ambiguous zone: 150-999 with a person-like word
        if 150 <= amt <= 999:
            words = [t for t in desc_tokens if t.isalpha() and len(t) > 2]
            if words:
                return {'type': 'expense', 'amount': amt, 'description': desc,
                        'date': today, 'month': get_month(today),
                        'raw': text, 'confirm': True}
        
        return {'type': 'expense', 'amount': amt, 'description': desc,
                'date': today, 'month': get_month(today), 'raw': text}

    # ── TODO: no amount, free text ────────────────────────────────
    return {'type': 'todo', 'content': text, 'date': None, 'raw': text}


def parse_note(raw_input, today=None):
    """
    Main entry point. Handles comma-separated multiple entries.
    Person commands (ADD_PERSON, etc.) are matched on the WHOLE input first,
    so commas inside a payload don't trigger split.
    Returns list of parsed dicts.
    """
    if today is None:
        today = datetime.now().strftime('%Y-%m-%dT%H:%M:%S')
    
    raw_input = raw_input.strip()
    
    cmd = parse_person_command(raw_input)
    if cmd:
        return [cmd]
    
    parts = [p.strip() for p in raw_input.split(',') if p.strip()]
    results = []
    for part in parts:
        parsed = parse_single(part, today)
        if parsed:
            results.append(parsed)
    return results

print("Parser functions defined. KNOWN_PERSONS:", sorted(KNOWN_PERSONS))

## 4. Test Cases — Real Notes

In [ ]:
TODAY = "2026-05-02T10:00:00"

# No category field; KNOWN_PERSONS gates weight detection.
test_cases = [
    # ── EXPENSES ──────────────────────────────────────────
    ("petrol 500",                      'expense',  {'amount': 500,  'description': 'petrol'}),
    ("500 petrol",                      'expense',  {'amount': 500,  'description': 'petrol'}),
    ("groceries 200",                   'expense',  {'amount': 200,  'description': 'groceries'}),
    ("bore motor repair 7500",          'expense',  {'amount': 7500}),
    ("780 mutton biryani + lollipop",   'expense',  {'amount': 780}),
    ("Medplus Pampers 143",             'expense',  {'amount': 143}),
    ("home electricity bill 3560",      'expense',  {'amount': 3560}),
    ("bsnl broadband bill 707",         'expense',  {'amount': 707}),
    ("5600 airport drop car rental",    'expense',  {'amount': 5600}),
    # NEW: bare object + small number — must be expense, not weight (biscuit not in KNOWN_PERSONS)
    ("biscuit 20",                      'expense',  {'amount': 20,   'description': 'biscuit'}),
    ("milk 60",                         'expense',  {'amount': 60,   'description': 'milk'}),

    # ── LEDGER (known persons) ─────────────────────────────
    ("gave Maddy 5k",                   'ledger',   {'person': 'maddy',  'direction': 'gave',     'amount': 5000}),
    ("give Maddy 10k",                  'ledger',   {'person': 'maddy',  'direction': 'gave',     'amount': 10000}),
    ("Maddy returned 6k",               'ledger',   {'person': 'maddy',  'direction': 'received', 'amount': 6000}),
    ("received 3k from thenna",         'ledger',   {'person': 'thenna', 'direction': 'received', 'amount': 3000}),
    ("Maddy gave 5k",                   'ledger',   {'person': 'maddy',  'direction': 'received', 'amount': 5000}),
    ("gift Maddy 500",                  'expense',  {'amount': 500}),

    # ── LEDGER (unknown persons → option B nudge) ──────────
    ("lent Ravi 2000",                  'ledger',   {'person': 'ravi',   'direction': 'gave',     'amount': 2000, 'unknown_person': True}),
    ("sent Priya 1500",                 'ledger',   {'person': 'priya',  'direction': 'gave',     'amount': 1500, 'unknown_person': True}),
    ("got 5k from Mani",                'ledger',   {'person': 'mani',   'direction': 'received', 'amount': 5000, 'unknown_person': True}),

    # ── WEIGHTS ───────────────────────────────────────────
    ("jeevi 62",                        'weight',   {'person': 'jeevi',   'weight': 62.0}),
    ("jeevi 65.1",                      'weight',   {'person': 'jeevi',   'weight': 65.1}),
    ("prani 11.3",                      'weight',   {'person': 'prani',   'weight': 11.3}),
    ("murugan 65",                      'weight',   {'person': 'murugan', 'weight': 65.0}),
    ("jeevi 65.1 empty stomach",        'weight',   {'person': 'jeevi',   'weight': 65.1}),

    # ── PERSON COMMANDS ────────────────────────────────────
    ("ADD_PERSON: ravi",                 'person_command', {'op': 'ADD_PERSON',    'name': 'ravi'}),
    ("REMOVE_PERSON: priya",             'person_command', {'op': 'REMOVE_PERSON', 'name': 'priya'}),
    ("MODIFY_PERSON: maddy mahdi",       'person_command', {'op': 'MODIFY_PERSON', 'old_name': 'maddy', 'new_name': 'mahdi'}),
    ("add_person: ananya",               'person_command', {'op': 'ADD_PERSON',    'name': 'ananya'}),  # case-insensitive

    # ── TODOS ─────────────────────────────────────────────
    ("update Amit about MCP",           'todo',     {}),
    ("complete app development phase 1",'todo',     {}),
    ("order mushroom fried rice",       'todo',     {}),
    ("haircut on thursday",             'todo',     {}),
]

passed = 0
failed = 0

for raw, expected_type, expected_fields in test_cases:
    results = parse_note(raw, TODAY)
    if not results:
        print(f"✗ NO RESULT   | {raw}")
        failed += 1
        continue
    
    result = results[0]
    actual_type = result['type']

    if actual_type == expected_type:
        field_ok = all(result.get(k) == v for k, v in expected_fields.items())
        if field_ok:
            confirm_flag = " [confirm]" if result.get('confirm') else ""
            nudge_flag  = " [nudge]"   if result.get('unknown_person') else ""
            print(f"✓{confirm_flag}{nudge_flag:10} | {raw}")
            passed += 1
        else:
            print(f"✗ FIELD FAIL  | {raw}")
            print(f"   Expected: {expected_fields}")
            print(f"   Got:      { {k: result.get(k) for k in expected_fields} }")
            failed += 1
    else:
        print(f"✗ TYPE FAIL   | {raw}")
        print(f"   Expected: {expected_type}, Got: {actual_type}")
        failed += 1

print(f"\n{'='*50}")
print(f"Passed: {passed}/{len(test_cases)}")
print(f"Failed: {failed}/{len(test_cases)}")
pct = passed / len(test_cases) * 100
print(f"Score:  {pct:.1f}%")
print("EXIT CRITERIA: need >= 85%" )
print("PASS ✓" if pct >= 85 else "FAIL ✗ — fix parser before Notebook 3")

## 5. Multi-Entry Test

In [4]:

# Test comma-separated entries
multi_tests = [
    ("tomato 50, groceries 200, petrol 500", 3, ['expense', 'expense', 'expense']),
    ("52 jeevi, 12 prani",                   2, ['weight',  'weight']),
    ("update Amit about MCP, complete app phase 1", 2, ['todo', 'todo']),
    ("petrol 500, gave Maddy 2k",            2, ['expense', 'ledger']),
]

for raw, expected_count, expected_types in multi_tests:
    results = parse_note(raw, TODAY)
    actual_types = [r['type'] for r in results]
    ok = len(results) == expected_count and actual_types == expected_types
    print(f"{'✓' if ok else '✗'} {raw}")
    if not ok:
        print(f"  Expected {expected_count} entries of types {expected_types}")
        print(f"  Got      {len(results)} entries of types {actual_types}")


✗ tomato 50, groceries 200, petrol 500
  Expected 3 entries of types ['expense', 'expense', 'expense']
  Got      3 entries of types ['weight_confirm', 'expense', 'expense']
✓ 52 jeevi, 12 prani
✗ update Amit about MCP, complete app phase 1
  Expected 2 entries of types ['todo', 'todo']
  Got      2 entries of types ['todo', 'expense']
✓ petrol 500, gave Maddy 2k


## 6. Confirmation Toast Simulator

In [ ]:
def get_confirmation_message(parsed):
    """Generate the confirmation toast message shown to user."""
    t = parsed['type']
    if t == 'expense':
        flag = " [confirm?]" if parsed.get('confirm') else ""
        return f"₹{parsed['amount']:.0f} {parsed['description']} logged{flag}"
    elif t == 'ledger':
        verb = "Gave" if parsed['direction'] == 'gave' else "Received from"
        base = f"{verb} {parsed['person'].title()} ₹{parsed['amount']:.0f} logged"
        if parsed.get('unknown_person'):
            base += f". Tip: ADD_PERSON: {parsed['person']} to track future entries cleanly"
        return base
    elif t == 'weight':
        note = f" ({parsed['note']})" if parsed.get('note') else ""
        return f"{parsed['person'].title()} weight: {parsed['weight']}kg logged{note}"
    elif t == 'person_command':
        if 'error' in parsed:
            return f"⚠ {parsed['error']}"
        if parsed['op'] == 'ADD_PERSON':
            return f"(would) Add '{parsed['name']}' to people"
        if parsed['op'] == 'REMOVE_PERSON':
            return f"(would) Remove '{parsed['name']}' from people"
        if parsed['op'] == 'MODIFY_PERSON':
            return f"(would) Rename '{parsed['old_name']}' → '{parsed['new_name']}'"
    elif t == 'todo':
        return f"Todo added: {parsed['content']}"
    return "Entry logged"

sample_inputs = [
    "biscuit 20",                              # was weight_confirm; now expense
    "milk 60",                                 # expense
    "gave Maddy 2k",                           # known person ledger
    "lent Ravi 2000",                          # unknown person → option B nudge
    "Maddy gave 1k",                           # known person ledger (received)
    "prani 65.1 empty stomach",                # weight
    "update Amit about MCP & its work",        # todo
    "groceries 300",                           # ambiguous expense
    "ADD_PERSON: ravi",                        # person command
    "MODIFY_PERSON: maddy mahdi",              # rename
    "REMOVE_PERSON: priya",                    # remove
]

print("Confirmation toasts:")
for raw in sample_inputs:
    results = parse_note(raw, TODAY)
    for r in results:
        print(f"  Input: {raw!r:42} → {get_confirmation_message(r)}")